# 2026-10-21_3 MG 基因表达堆叠小提琴图（sharedY 版：同一基因胚胎/成年 y 轴一致）

**需求与口径**（在 2026-10-21_2 基础上修改）

- **同一基因在胚胎页与成年页的 y 轴范围保持一致**（便于两期对比）
- 采用**每基因共享**而非全图单一刻度：各基因 y 轴上限取该基因跨两期的全局最大值；若全图共用一个刻度（上限 ≈ 5.3），COTL1（上限 2.95）等低表达基因会被压扁，而跨基因的量级对比已由 2026-10-21_1 dotplot 覆盖
- 其余口径不变：仅 MG、剔除 70years；X 轴 Early-central; Early-peripheral; Mid-central; Mid-peripheral; Late-central; Late-peripheral（成年页 young/old 同序）；基因堆叠 ABCB11 顶部；配色 stage 本色 = central、浅色 = peripheral；胚胎/成年分页；仅 PDF

**实现方式**

- 对每个基因计算跨两期全局最大值 `gene_y_max`，通过 `geom_blank`（每基因一行）把两页该基因面板的 y 上限撑到 `gene_y_max + 余量`
- 用 `scale_y_continuous(limits = c(0, NA), oob = squish)` 替换 Seurat 默认 y 标度：下限统一为 0（`add.noise` 造成的 ±1e-5 抖动被压回 0），保证两页范围**完全一致**
- 替换标度时 ggplot 提示 `Scale for y is already present. Adding another scale for y, which will replace the existing scale.` 属预期行为
- notebook 内以 `stopifnot` 校验两页每个基因的 y 轴范围逐一相等

**数据源（复用每细胞表，不重读大对象）**

- `Data/2026-10-21_1_MG_gene_percell_expr.rds`

**输出**

- `Plot/2026-10-21_3_MG_gene_expression_stacked_violin_sharedY.pdf`（2 页：胚胎 / 成年）

In [ ]:
# ---- 1. 环境与参数 ----------------------------------------------------------
suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
  library(ggplot2)
  library(scales)
})
set.seed(1)

DATE_STR <- "2026-10-21_3"
PROJ_DIR <- "/project/work/2026-10-21_violinplot"
DATA_DIR <- file.path(PROJ_DIR, "Data")
PLOT_DIR <- file.path(PROJ_DIR, "Plot")

PERCELL_RDS <- file.path(DATA_DIR, "2026-10-21_1_MG_gene_percell_expr.rds")
stopifnot(file.exists(PERCELL_RDS))

GENES <- c("ABCB11", "TTC39B", "TFRC", "DIO2", "PITPNC1", "COTL1", "APOE")

GROUP_LEVELS_EMB <- c("Early-central", "Early-peripheral", "Mid-central",
                      "Mid-peripheral", "Late-central", "Late-peripheral")
GROUP_LEVELS_ADU <- c("young-central", "young-peripheral", "old-central", "old-peripheral")

# ---- 配色（沿用 panelFH 的 stage/region 逻辑；用户提供）-----------------------
cols.stage.full <- c('#59A14F','#EDC948','#B07AA1','#4E79A7','#E15759')  ## 用户指定统一配色
names(cols.stage.full) <- c('Early','Mid','Late','Young','Old')
cols.stage      <- cols.stage.full[c("Early","Mid","Late")]                       ## dev: Early/Mid/Late
cols.adultstage <- setNames(cols.stage.full[c("Young","Old")], c("young","old"))  ## adult: 数据里为小写 young/old
lighten_col <- function(col, f = 0.45) {
  m <- grDevices::col2rgb(col)
  grDevices::rgb(t(m + (255 - m) * f), maxColorValue = 255)
}
## central = 本色, peripheral = 浅色
region_cols <- function(base) {
  v <- c(base, sapply(base, lighten_col))
  names(v) <- c(paste0(names(base), "_central"), paste0(names(base), "_peripheral"))
  v
}

cols.emb.groups <- region_cols(cols.stage)
names(cols.emb.groups) <- gsub("_", "-", names(cols.emb.groups))
cols.adu.groups <- region_cols(cols.adultstage)
names(cols.adu.groups) <- gsub("_", "-", names(cols.adu.groups))

cat("胚胎分组配色：\n"); print(cols.emb.groups[GROUP_LEVELS_EMB])
cat("\n成年分组配色：\n"); print(cols.adu.groups[GROUP_LEVELS_ADU])

In [ ]:
# ---- 2. 读取每细胞表，计算每基因跨两期全局 y 上限 -----------------------------
percell <- readRDS(PERCELL_RDS)
percell$period <- factor(as.character(percell$period), levels = c("Embryonic", "Adult"))
emb <- percell[percell$period == "Embryonic", ]
adu <- percell[percell$period == "Adult", ]
cat("胚胎 MG 细胞:", nrow(emb), "| 成年 MG 细胞:", nrow(adu), "\n")
cat("\n[Embryonic] 分组细胞数：\n"); print(table(droplevels(factor(as.character(emb$stage_region), levels = GROUP_LEVELS_EMB))))
cat("\n[Adult] 分组细胞数：\n"); print(table(droplevels(factor(as.character(adu$stage_region), levels = GROUP_LEVELS_ADU))))

## 每基因跨两期（胚胎+成年全部 MG 细胞）的 lognorm 最大值 = 两页共同的 y 上限基准
gene_y_max <- vapply(GENES, function(g) max(percell[[paste0(g, "_lognorm")]]), numeric(1))
cat("\n每基因跨两期全局最大 lognorm：\n"); print(round(gene_y_max, 3))
cat("\n胚胎期每基因最大值：\n")
print(round(vapply(GENES, function(g) max(emb[[paste0(g, "_lognorm")]]), numeric(1)), 3))
cat("\n成年期每基因最大值：\n")
print(round(vapply(GENES, function(g) max(adu[[paste0(g, "_lognorm")]]), numeric(1)), 3))

build_mini_seurat <- function(df, group_levels) {
  counts_mat <- Matrix::t(as(as.matrix(df[, paste0(GENES, "_counts")]), "dgCMatrix"))
  data_mat   <- Matrix::t(as(as.matrix(df[, paste0(GENES, "_lognorm")]), "dgCMatrix"))
  rownames(counts_mat) <- GENES; colnames(counts_mat) <- df$cell
  rownames(data_mat)   <- GENES; colnames(data_mat)   <- df$cell
  mini_meta <- data.frame(
    stage_region = factor(as.character(df$stage_region), levels = group_levels),
    row.names = df$cell
  )
  mini <- CreateSeuratObject(counts = counts_mat, meta.data = mini_meta)
  mini[["RNA"]]$data <- data_mat
  Idents(mini) <- "stage_region"
  mini
}

## 追加版本（2026-10-21）：按 donor_age 分组（`_byAge` 后缀）

- x 轴不按 stage 而按 **donor_age**：相同 age 的 central / peripheral 相邻（age-central, age-peripheral 交替），
  age 顺序按用户指定：胚胎 70/79/87/91/100/103/116/137/141/142/162/165 days（12 龄）；
  成年 21/25/30/32/67/70/75/86 years（8 龄）
- **纳入 70years**（原 _1/_3 口径剔除的 Old_unpaired）：该龄仅 macula（central），peripheral 保留空槽位
  （与 2026-10-21_4 的空槽位处理一致）；其余口径不变（仅 MG、7 基因）
- 配色沿用 stage 五色：各 age 继承其 stage 颜色（central = 本色，peripheral = 浅色 45%），
  与 stage 版可直接对照；因 x 标签已完整标识各分组，**省略填充图例**（颜色含义写入副标题）
- 胚胎 / 成年分页；同一基因两页 y 轴一致（sharedY，上限跨两期含 70years 全局计算）；x 标签两行（age + region）
- 输出：`Plot/2026-10-21_3_MG_gene_expression_stacked_violin_sharedY_byAge.pdf`（2 页，14×8 英寸）；
  **原 stage 版 `_sharedY.pdf` 不改动**；每细胞表（含 70years）另存
  `Data/2026-10-21_3_MG_gene_percell_expr_with70y.rds`

In [ ]:
# ---- 6. byAge 追加版：补提取 70years MG 细胞（原口径剔除的 Old_unpaired）----
# 70years 仅 macula（central）、无 peripheral 对照；byAge 版将其纳入，
# "70years-peripheral" 保留空槽位
ADU_RDS_AGE <- "/project/work/2026-05-25_create_summary/Data/2026-06-05_new_adult_human_harmony_anno.rds"
stopifnot(file.exists(ADU_RDS_AGE))

obj70 <- readRDS(ADU_RDS_AGE)
stopifnot(all(GENES %in% rownames(obj70)))
meta70 <- CONTACT_EMAIL
keep70 <- which(meta70$celltype == "MG" & as.character(meta70$donor_age) == "70years")
cells70 <- rownames(meta70)[keep70]
m70 <- meta70[keep70, , drop = FALSE]
cat("70years MG 细胞数:", length(cells70), "\n")
print(table(tolower(as.character(m70$tissue))))

tissue70 <- tolower(as.character(m70$tissue))
region70 <- ifelse(tissue70 %in% c("fovea", "macula", "central"), "central",
                   ifelse(tissue70 == "peripheral", "peripheral", NA_character_))
stopifnot(!any(is.na(region70)), all(region70 == "central"))

counts70 <- GetAssayData(obj70, assay = "RNA", layer = "counts")[GENES, cells70]
data70   <- GetAssayData(obj70, assay = "RNA", layer = "data")[GENES, cells70]
counts_df70  <- as.data.frame(t(as.matrix(counts70)))
lognorm_df70 <- as.data.frame(t(as.matrix(data70)))
colnames(counts_df70)  <- paste0(GENES, "_counts")
colnames(lognorm_df70) <- paste0(GENES, "_lognorm")

y70 <- data.frame(
  cell = cells70, period = "Adult", celltype = "MG",
  donor_age = "70years", tissue = as.character(m70$tissue),
  region = region70, stage = "old", stage_region = "old-central",
  nCount_RNA = as.numeric(m70$nCount_RNA),
  stringsAsFactors = FALSE
)
y70 <- cbind(y70, counts_df70, lognorm_df70)
rm(obj70, meta70, m70, counts70, data70); invisible(gc())

## 合并（percell 来自本 notebook cell 2，不含 70years）
percell_age <- rbind(percell, y70)
percell_age$period <- factor(percell_age$period, levels = c("Embryonic", "Adult"))
percell_age$age_region <- paste0(percell_age$donor_age, "\n", percell_age$region)
cat("\n含 70years 的 MG 总细胞数:", nrow(percell_age), "\n")
print(table(percell_age$period, percell_age$donor_age == "70years"))

percell_age_rds <- file.path(DATA_DIR, "2026-10-21_3_MG_gene_percell_expr_with70y.rds")
saveRDS(percell_age, percell_age_rds)
cat("已保存:", basename(percell_age_rds), "\n")